In [5]:
import pandas as pd
import os

In [52]:
# Leggiamo tutte quante le XAI
os.listdir("txts")

['microarray.txt',
 '.ipynb_checkpoints',
 'BCELLS_CSF_NOCLUSTER.txt',
 'BCELLS_CSF_CLUSTER.txt',
 'BCELLS_CSF_80%_CLUSTER.txt',
 'BCELLS_PBMC_NOCLUSTER.txt',
 'BCELLS_PBMC_CLUSTER.txt',
 'BCELLS_PBMC_80%_CLUSTER.txt',
 'CD4_CSF_80%_NOCLUSTER.txt',
 'CD4_CSF_80%_CLUSTER.txt',
 'BCELLS_PBMC_80%.txt']

In [64]:
def read_cluster_txt(path, file):
    with open(os.path.join(path, file), "r") as f:
        content = f.read()
    content = content.split(",")
    content = list(map(lambda x: x.strip(),content))
    return content

In [95]:
def create_gene_dataframe(directory="txts"):
    full_lists = {}
    all_genes = set()
    for i in os.listdir(directory):
        if ".txt" in i:
            content = read_cluster_txt(directory, i)
            all_genes = all_genes.union(set(content)) #manteniamo una lista di tutti i geni visti almeno una volta
            
            if i == "microarray.txt":
                full_lists["microarray"] = content
                continue
            label = i.split("_")
            subtype = label[0]
            tissue = label[1]
            type_cluster = ("_".join(label[2:])).replace(".txt","")
            tuple_label = (subtype, tissue, type_cluster)
            
            full_lists[tuple_label] = content
    
    #lets create df
    dataframe = pd.DataFrame(index = list(all_genes))
   # Add columns
    for cluster_key, gene_list_for_cluster in full_lists.items():
        # For each gene in the DataFrame's index, check if it's in the current cluster's gene list
        # Convert boolean (True/False) to integer (1/0)
        dataframe[cluster_key] = dataframe.index.isin(
            gene_list_for_cluster
        ).astype(int)
        
    return dataframe
        

In [101]:
df = create_gene_dataframe()
df.to_csv("xai_summary.csv")

In [108]:
def find_gene_in_at_least_n_cluster(df,min_cluster):

    gene_cluster_counts = df.sum(axis=1)

    # Filter genes where the count is >= min_clusters
    # Your prompt "sum of row > 4" would be gene_cluster_counts > 4.
    # "at least 4" means gene_cluster_counts >= 4.
    genes_meeting_criteria = gene_cluster_counts[
        gene_cluster_counts >= min_cluster
    ]
    return genes_meeting_criteria